## Word2Vec as an Autoencoder: Learning Word Representations (20 Newsgroups)

An autoencoder squeezes its input through a narrow bottleneck and is trained
to rebuild something from the code. In the previous notebook the input was
an image and the target was *the same image*. Word2Vec (skip-gram) keeps the
exact same shape -- a wide input, a narrow code, a wide output -- and changes
only the target:

$$
\underbrace{\text{one-hot word } (V)}_{\text{input}}
\;\xrightarrow{\;\text{encoder}\;}\;
\underbrace{h \in \mathbb{R}^{d}}_{\text{bottleneck, } d \ll V}
\;\xrightarrow{\;\text{decoder}\;}\;
\underbrace{P(\text{context word} \mid \text{word}) \text{ over } V}_{\text{output}}
$$

Instead of reconstructing the word itself (which would be trivial -- just
copy it), the network must *predict the words that appear around it*. To do
that well through a $d$-dimensional bottleneck, words that show up in similar
contexts have to be given similar codes. That code is the word vector.

No labels are involved: the training signal is just raw text. We

1. build the encoder / decoder and train it on unlabeled newsgroup posts,
2. inspect what the codes learned -- nearest neighbours and a 2-D map,
3. put the word vectors to work on a downstream task: **classifying whole
   documents into 20 topics with only a handful of labeled examples**,
   compared against a baseline that has no pretraining.

### 1. Setup

Standard imports, seed, and device -- same conventions as the rest of this series.

In [ ]:
import re
import collections
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

In [ ]:
# Ensure reproducible results
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
# Set device to GPU if available, otherwise fallback to CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### 2. Dataset: 20 Newsgroups

About 18k Usenet posts from 20 topical newsgroups (religion, space,
hockey, cryptography, cars, ...), already split into train (11.3k) and test
(7.5k) posts. We strip the headers, signatures and quoted replies so the
model has to work from the actual message text rather than from give-away
metadata such as the newsgroup name.

The word vectors are trained on the **train text only, ignoring the labels**.
The labels reappear only in the last section, for the downstream task.

In [ ]:
# Fetch the 20 newsgroups dataset, removing metadata headers, footers, and quotes to avoid shortcuts
REMOVE_METADATA = ("headers", "footers", "quotes")

train_raw = fetch_20newsgroups(subset="train", remove=REMOVE_METADATA, data_home="data")
test_raw = fetch_20newsgroups(subset="test", remove=REMOVE_METADATA, data_home="data")

class_names = train_raw.target_names
y_train, y_test = train_raw.target, test_raw.target

print(f"Train posts: {len(train_raw.data):,}")
print(f"Test posts: {len(test_raw.data):,}")
print(f"Number of classes: {len(class_names)}")
print("\nClass Names:", class_names)

In [ ]:
for i in [0, 3]:
    print(f"CLASS: [{class_names[y_train[i]]}]")
    print(train_raw.data[i][:350].strip(), "...")
    print("="*20)

### 3. From text to (word, context) training pairs

**Tokenise and build a vocabulary.** Lower-case, keep alphabetic tokens, and
keep only words that occur at least 40 times in the training text. That leaves
a vocabulary of roughly 5,000 words; everything rarer is dropped (there is
too little evidence to learn a good vector for a word seen twice).

The decoder produces one score per vocabulary word, so a small $V$ keeps the
notebook fast on a CPU.

In [ ]:
def tokenize(text):
    """Convert text to lowercase and extract alphabetic word tokens."""
    return re.findall(r"[a-z]+", text.lower())

# Tokenize all documents
train_docs = [tokenize(doc) for doc in train_raw.data]
test_docs = [tokenize(doc) for doc in test_raw.data]

In [ ]:
# Filter vocabulary: keep only words appearing at least 40 times in train set
MIN_COUNT = 40
counts = collections.Counter(word for doc in train_docs for word in doc)
vocab = [word for word, count in counts.most_common() if count >= MIN_COUNT]
word2id = {word: idx for idx, word in enumerate(vocab)}
V = len(vocab)

n_tokens = sum(counts.values())
n_kept = sum(counts[word] for word in vocab)

print(f"Raw tokens: {n_tokens:,} | Distinct words: {len(counts):,}")
print(f"Vocabulary size (count >= {MIN_COUNT}): V = {V:,}")
print(f"Coverage: {n_kept / n_tokens:.1%} of all tokens")
print("Top 12 most frequent words:", vocab[:12])

**Skip-gram pairs.** For every word in a document (the *centre* word) we
pair it with each neighbour within a window of $\pm 3$ positions (the
*context* words). Each pair is one training example: input = centre word,
target = context word. A tiny example, window $\pm 2$:

In [ ]:
toy = "the shuttle reached orbit after a clean launch".split()
WINDOW_TOY = 2
centre_pos = toy.index("orbit")
context = [toy[j] for j in range(max(0, centre_pos - WINDOW_TOY), min(len(toy), centre_pos + WINDOW_TOY + 1))
           if j != centre_pos]
print("sentence:", " ".join(toy))
print(f"centre word 'orbit' -> training pairs: {[('orbit', c) for c in context]}")

**Subsampling frequent words.** Words like *the* or *of* appear in almost every
window but say little about their neighbours, and they would dominate the
pairs. Following the original word2vec paper, each occurrence of word $w$ is
kept with probability

$$
P(\text{keep } w) = \min\!\Big(1,\; \sqrt{t / f(w)} + t / f(w)\Big),
$$

where $f(w)$ is its share of all tokens and $t$ a small threshold ($10^{-4}$).
Rare words are always kept; very frequent ones are thinned out heavily. Since
the subsampling is random, we redraw it every epoch, so the model sees
different (word, context) pairs each time around.

The pair generator below is vectorised: for each offset $k = 1, \dots, 3$
it pairs every token with the one $k$ places to its right (and the mirrored
pair), skipping pairs that straddle two documents.

In [ ]:
# Create a single continuous stream of token IDs while tracking their original document source
token_ids, doc_ids = [], []
for doc_idx, doc in enumerate(train_docs):
    valid_ids = [word2id[word] for word in doc if word in word2id]
    token_ids.extend(valid_ids)
    doc_ids.extend([doc_idx] * len(valid_ids))

token_ids = np.array(token_ids)
doc_ids = np.array(doc_ids)

# Configure sliding window and subsampling threshold
WINDOW = 3
SUBSAMPLE_T = 1e-4

# Calculate word frequencies to compute probability of keeping each word
word_counts = np.bincount(token_ids, minlength=V)
freq = word_counts / len(token_ids)
p_keep = np.minimum(1.0, np.sqrt(SUBSAMPLE_T / freq) + SUBSAMPLE_T / freq)

def make_pairs(seed):
    """Generates randomized (centre, context) training pairs after applying subsampling."""
    rng = np.random.RandomState(seed)
    keep_mask = rng.rand(len(token_ids)) < p_keep[token_ids]
    filtered_tokens = token_ids[keep_mask]
    filtered_docs = doc_ids[keep_mask]

    centres, contexts = [], []
    for offset in range(1, WINDOW + 1):
        # Ensure we only pair tokens belonging to the same document
        same_doc_mask = filtered_docs[:-offset] == filtered_docs[offset:]

        centres.append(filtered_tokens[:-offset][same_doc_mask])
        contexts.append(filtered_tokens[offset:][same_doc_mask])

        # Add mirror pairs (context to centre relationship is symmetric)
        centres.append(filtered_tokens[offset:][same_doc_mask])
        contexts.append(filtered_tokens[:-offset][same_doc_mask])

    return np.concatenate(centres), np.concatenate(contexts)

# Demo run of generating pairs
centre_ids, context_ids = make_pairs(seed=0)
print(f"Tokens in stream: {len(token_ids):,}")
print(f"Training pairs generated: {len(centre_ids):,}")
print("Sample training pairs:", [(vocab[c_id], vocab[ctx_id]) for c_id, ctx_id in zip(centre_ids[:5], context_ids[:5])])
print("Probability of keeping common words:", {w: round(float(p_keep[word2id[w]]), 3) for w in ["the", "of", "you", "nasa", "shuttle"]})

### 4. The model: an encoder-decoder with a $d$-dimensional bottleneck

Same structure as the image autoencoder, with the same two pieces:

- **`Encoder`**: $V \to d$. Multiplying a one-hot vector by a $V \times d$
  weight matrix just selects one row of it, so we implement it as
  `nn.Embedding` (a row lookup) -- mathematically identical, without ever
  building the sparse one-hot vector. **The rows of this matrix are the word
  vectors**, and they are the thing we care about.
- **`Decoder`**: $d \to V$. A single linear layer producing one score
  (logit) per vocabulary word; softmax turns them into a probability
  distribution over the context word.

There is no non-linearity in between, so the whole thing is a low-rank
factorisation of the word-to-context relationship -- which is why it is so
cheap, and still works.

In [ ]:
EMBED_DIM = 64

class Encoder(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        # An embedding layer acts as a look-up table mapping word IDs to continuous vectors
        self.emb = nn.Embedding(vocab_size, dim)

    def forward(self, word_ids):
        return self.emb(word_ids)


class Decoder(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        # Maps our low-dimensional space back to a linear score for each vocabulary word
        self.out = nn.Linear(dim, vocab_size)

    def forward(self, h):
        return self.out(h)


class Word2VecAutoencoder(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.encoder = Encoder(vocab_size, dim)
        self.decoder = Decoder(vocab_size, dim)

    def forward(self, word_ids):
        h = self.encoder(word_ids)
        return self.decoder(h), h

# Instantiate and count model parameters
model = Word2VecAutoencoder(V, EMBED_DIM)
total_params = sum(p.numel() for p in model.parameters())
print(f"V = {V}, d = {EMBED_DIM} | Total Trainable Parameters: {total_params:,}")

Sanity check of the claim that the embedding lookup *is* the encoder
applied to a one-hot input:

In [ ]:
# Map text keys back to embedding vector index space
word_ids = torch.tensor([word2id["space"], word2id["god"]])
one_hot = nn.functional.one_hot(word_ids, num_classes=V).float()

# Mathematically verify that lookups are equivalent to sparse matrix multiplication
via_matmul = one_hot @ model.encoder.emb.weight
via_lookup = model.encoder(word_ids)

print("One-hot Matrix Multiplication == Embedding Table Lookup:", torch.allclose(via_matmul, via_lookup))

### Alternative: Word2Vec Encoder with a Bare DNN Layer (`nn.Linear`)

Below, we define an alternative Encoder class `DNNDecoderEncoder` that uses a standard dense linear layer `nn.Linear` instead of an embedding lookup table.

To feed word IDs into this linear layer, we must first convert them into one-hot vectors of size $V$.

In [ ]:
class DNNEncoder(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        # A linear layer without bias maps a V-dimensional one-hot vector to a d-dimensional continuous vector
        self.linear = nn.Linear(vocab_size, dim, bias=False)
        self.vocab_size = vocab_size

    def forward(self, word_ids):
        # 1. Convert word indices to one-hot vectors: shape (batch_size, V)
        one_hot = nn.functional.one_hot(word_ids, num_classes=self.vocab_size).float()
        # 2. Pass through the bare dense linear layer
        return self.linear(one_hot)

# Instantiate both styles with the same vocabulary size and dimension
torch.manual_seed(0)
embedding_encoder = Encoder(V, EMBED_DIM)

torch.manual_seed(0)
dnn_encoder = DNNEncoder(V, EMBED_DIM)

# Copy weights from the embedding layer to the linear layer to ensure they are identical
with torch.no_grad():
    dnn_encoder.linear.weight.copy_(embedding_encoder.emb.weight.t())

# Verify outputs with sample word IDs
sample_ids = torch.tensor([word2id["space"], word2id["god"]])
out_embedding = embedding_encoder(sample_ids)
out_dnn = dnn_encoder(sample_ids)

print("Embedding output shape:", out_embedding.shape)
print("Bare DNN Linear output shape:", out_dnn.shape)
print("Are the representations mathematically identical?", torch.allclose(out_embedding, out_dnn))

### Extending to a Multi-Layer Bottleneck Encoder

If we want to build a deeper encoder instead of a single linear mapping, we can define a **Multi-Layer DNN Encoder**. This setup maps our sparse one-hot input vector through one or more hidden layers with non-linear activations (like `nn.ReLU`) before arriving at the final $d$-dimensional bottleneck representation.

In [ ]:
class MultiLayerDNNEncoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim, bottleneck_dim):
        super().__init__()
        self.vocab_size = vocab_size

        # Multi-layer feedforward architecture (MLP)
        self.network = nn.Sequential(
            nn.Linear(vocab_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim)
        )

    def forward(self, word_ids):
        # 1. Convert word indices to one-hot vectors
        one_hot = nn.functional.one_hot(word_ids, num_classes=self.vocab_size).float()
        # 2. Pass through the multi-layer neural network
        return self.network(one_hot)

# Instantiate a 2-layer deep encoder
hidden_dimension = 128
multi_layer_encoder = MultiLayerDNNEncoder(V, hidden_dimension, EMBED_DIM)

# Verification with our sample word IDs
sample_outputs = multi_layer_encoder(sample_ids)
print("Multi-layer DNN Encoder output shape:", sample_outputs.shape)

### 5. Loss function and training utilities

The output is a distribution over $V$ words and the target is one particular
context word, so the loss is **cross-entropy** -- a full softmax over the whole
vocabulary. (The original word2vec replaces this with *negative sampling* to
scale to vocabularies of millions of words; with $V \approx 5{,}000$ the full
softmax is affordable and keeps the "encoder-decoder" picture exact.)

Two things differ from the image autoencoder:

- Each epoch draws a fresh subsample of pairs (Section 3), so we call
  `make_pairs` inside the loop.
- The loss will **not** approach zero. A single word does not determine its
  neighbours -- *space* can be followed by *shuttle*, *station*, *program*... --
  so even a perfect model has an irreducible loss. A model that knew nothing
  would score $\ln V$; the aim is to get clearly below that.

In [ ]:
def train_word2vec(model, epochs, batch_size=4096, lr=3e-3):
    """Trains the Word2Vec autoencoder model using Cross-Entropy loss."""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"step_loss": []}
    for epoch in range(epochs):
        model.train()
        # Regenerate pairs with new random subsampling each epoch
        centres, contexts = make_pairs(seed=epoch)

        # Shuffle dataset pairs
        order = np.random.permutation(len(centres))
        centres = torch.from_numpy(centres[order])
        contexts = torch.from_numpy(contexts[order])

        running_loss, n_batches = 0.0, 0
        for start in range(0, len(centres), batch_size):
            xb = centres[start:start + batch_size].to(device)
            yb = contexts[start:start + batch_size].to(device)

            optimizer.zero_grad()
            logits, _ = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            history["step_loss"].append(loss.item())
            running_loss += loss.item()
            n_batches += 1

        epoch_loss = running_loss / n_batches
        print(f"Epoch {epoch + 1}/{epochs} | Average Train Loss: {epoch_loss:.4f} ({len(centres):,} pairs)")

    return history

### 6. Experiment 1: train the word2vec autoencoder

3 epochs of Adam over ~5.5M (word, context) pairs each. On a CPU this takes a
few minutes (about 2 minutes per epoch) -- lower `EPOCHS_W2V` to get a
quicker, rougher result.

In [ ]:
EPOCHS_W2V = 10

torch.manual_seed(0)
np.random.seed(0)

w2v = Word2VecAutoencoder(V, EMBED_DIM)

# Theoretical baseline: loss of a random prediction distribution
random_baseline_loss = np.log(V)
print(f"Baseline loss for completely random uniform guess: ln(V) = {random_baseline_loss:.3f}")
print("\n--- Training word2vec autoencoder (unsupervised task) ---")
w2v_history = train_word2vec(w2v, epochs=EPOCHS_W2V)

In [ ]:
# Compute running averages to smooth the noisy per-step optimization metrics
step_loss = np.array(w2v_history["step_loss"])
window_size = 100
smooth_loss = np.convolve(step_loss, np.ones(window_size) / window_size, mode="valid")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(step_loss, color="#2a78d6", linewidth=0.5, alpha=0.3, label="raw step loss")
ax.plot(np.arange(len(smooth_loss)) + window_size // 2, smooth_loss, color="#2a78d6", linewidth=2, label="100-step moving average")
ax.axhline(np.log(V), color="gray", linestyle="--", linewidth=1, label=r"$\ln V$: Random Uniform baseline")

ax.set_xlabel("Training Step")
ax.set_ylabel("Cross-entropy Loss")
ax.set_title("Word2Vec Autoencoder Training Progression")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

### 7. What did the bottleneck learn?

The loss tells us the model got better at predicting context; what we
actually want to know is whether the 64-d codes are *meaningful*. We never
told the model what any word means, so two checks:

**Nearest neighbours.** For a query word, list the words whose vectors have
the highest cosine similarity.

In [ ]:
# Extract embedding representations for cosine calculation
word_vectors = w2v.encoder.emb.weight.detach().cpu().numpy()
unit_vectors = normalize(word_vectors)

def nearest(word, k=7):
    """Retrieves top-k closest words according to cosine similarity metrics."""
    if word not in word2id:
        return [f"'{word}' is not in vocabulary"]
    similarity_scores = unit_vectors @ unit_vectors[word2id[word]]
    closest_indices = np.argsort(-similarity_scores)[1:k+1]
    return [vocab[idx] for idx in closest_indices]

print("=== Exploring Closest Neighborhoods ===")
for query_word in ["god", "space", "hockey", "windows", "gun", "israel", "encryption", "car", "price"]:
    print(f"{query_word:>11} -> {', '.join(nearest(query_word))}")

The neighbours are topically coherent -- *space* sits with *shuttle, nasa,
jpl, lunar*; *hockey* with *teams, season, league*; *car* with *miles, cars,
wheels*. Nothing here is a dictionary definition: the only thing the model saw was which
words occur near which. (Some neighbours are quirks of this corpus and
of being trained on only ~2M tokens; word2vec vectors trained on billions of
words are considerably sharper.)

**A 2-D map.** Project the vectors of a few hand-picked words from different
topics with t-SNE. Topic labels are used only to colour the points.

In [ ]:
topics = {
    "religion":  ["god", "jesus", "bible", "church", "christian", "faith", "prayer", "sin"],
    "space":     ["nasa", "shuttle", "orbit", "launch", "moon", "satellite", "lunar", "spacecraft"],
    "sports":    ["hockey", "baseball", "team", "season", "game", "players", "league", "pitcher"],
    "computers": ["windows", "dos", "disk", "software", "drive", "monitor", "graphics", "printer"],
    "guns":      ["gun", "guns", "weapons", "firearms", "handgun", "rifle", "crime", "police"],
    "mideast":   ["israel", "arab", "jews", "turkish", "armenian", "lebanese", "palestinian", "muslim"],
    "cars/bikes":  ["car", "engine", "bike", "cars", "oil", "brake", "riding", "dealer"],
}

topic_colors = dict(zip(topics, ["#2a78d6", "#eb6834", "#2ca25f", "#8856a7", "#d62728", "#17becf", "#7f7f7f"]))

selected_words, colors, labels = [], [], []
for topic_name, word_list in topics.items():
    for w in word_list:
        if w in word2id:
            selected_words.append(w)
            colors.append(topic_colors[topic_name])
            labels.append(topic_name)
        else:
            print(f"(Skipping '{w}': not in the vocabulary)")

# Project selected vectors using t-SNE
points_nd = unit_vectors[[word2id[w] for w in selected_words]]
points_2d = TSNE(n_components=2, random_state=0, init="pca", perplexity=10).fit_transform(points_nd)

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(points_2d[:, 0], points_2d[:, 1], c=colors, s=60, edgecolors='none', alpha=0.85)

# Annotate the mapped words onto the scatter-plot canvas
for (x_coord, y_coord), text in zip(points_2d, selected_words):
    ax.annotate(text, (x_coord, y_coord), xytext=(5, 3), textcoords="offset points", fontsize=9, alpha=0.95)

# Construct legend labels visually
for topic_name, col in topic_colors.items():
    ax.scatter([], [], c=col, label=topic_name)

ax.legend(frameon=False, loc="best", fontsize=9)
ax.set_title("2D Semantic Clustering of Word Embeddings via t-SNE", fontsize=12, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()
plt.show()

Words from the same topic land near each other. This is the same picture as
the digit clusters in the image autoencoder: an objective that never
mentions the "classes" still organises the code space by them. The next
section puts that structure to work.

### 8. Putting the word vectors to work: document classification with few labels

Task: given a whole post, predict which of the 20 newsgroups it came from
(chance = 5%). The catch is the same as in the image notebook -- **labels are
scarce**: we sweep the number of labeled posts per class from 1 up to 100,
while the word vectors were learned from thousands of unlabeled posts.

Three models, each a plain multinomial logistic regression on top of a
different document representation, and all trained on the *same* small
labeled subset and evaluated on the full 7.5k-post test set:

- **Word2Vec (pretrained)** -- a document is the **average of its word
  vectors** (from the encoder above, frozen), length-normalised. A
  64-number summary of the post.
- **TF-IDF bag of words (no pretraining)** -- each post is a sparse ~5,000-d
  vector of word weights over the same vocabulary. Words carry no notion of
  similarity: *hockey* and *nhl* are as unrelated as *hockey* and *modem*, so
  the classifier can only learn each word's weight from the labeled posts.
- **Random embeddings, averaged** -- the same 64-d averaging pipeline, but
  with an untrained (randomly initialised) encoder. This controls for the
  possibility that the gain comes from the averaging rather than from
  what was learned.

Each labeled-set size is repeated with 5 different random subsets, because
with a handful of examples the particular ones drawn matter a lot.

In [ ]:
def doc_to_ids(docs):
    """Converts raw doc tokens to array of vocabulary indices."""
    return [np.array([word2id[w] for w in doc if w in word2id], dtype=int) for doc in docs]


def average_embeddings(docs, vectors):
    """Averages the word vectors in each document, and length-normalizes the result."""
    feats = np.stack([vectors[ids].mean(axis=0) if len(ids) else np.zeros(vectors.shape[1])
                      for ids in doc_to_ids(docs)])
    return normalize(feats)


# Generate random control embeddings
torch.manual_seed(0)
random_vectors = Word2VecAutoencoder(V, EMBED_DIM).encoder.emb.weight.detach().numpy()

# Setup the baseline TF-IDF Vectorizer
tfidf = TfidfVectorizer(vocabulary=vocab, token_pattern=r"[a-z]+", sublinear_tf=True)
tfidf.fit(train_raw.data)

# Package our comparative representations
features = {
    "w2v":    (average_embeddings(train_docs, word_vectors), average_embeddings(test_docs, word_vectors)),
    "random": (average_embeddings(train_docs, random_vectors), average_embeddings(test_docs, random_vectors)),
    "tfidf":  (tfidf.transform(train_raw.data), tfidf.transform(test_raw.data)),
}

for representation_name, (Xtr, Xte) in features.items():
    print(f"{representation_name:>8}: Train representation shape = {Xtr.shape} | Test shape = {Xte.shape}")

In [ ]:
def sample_labeled_subset(targets, n_per_class, seed):
    """Extracts a balanced class split dynamically based on indices."""
    rng = np.random.RandomState(seed)
    sampled_indices = []
    for class_idx in range(len(class_names)):
        class_pool = np.where(targets == class_idx)[0]
        sampled_indices.append(rng.choice(class_pool, size=n_per_class, replace=False))
    return np.concatenate(sampled_indices)


def fit_and_score(rep_name, idx):
    """Trains a logistic regression classifier on a subset of representation and returns test accuracy."""
    Xtr, Xte = features[rep_name]
    clf = LogisticRegression(C=10, max_iter=2000).fit(Xtr[idx], y_train[idx])
    return clf.score(Xte, y_test)

N_PER_CLASS = [1, 2, 5, 10, 20, 50, 100]
N_SEEDS = 5

model_names = {
    "w2v": "Word2Vec (pretrained, averaged)",
    "tfidf": "TF-IDF bag of words (no pretraining)",
    "random": "random embeddings, averaged"
}

acc = {name: np.zeros((len(N_PER_CLASS), N_SEEDS)) for name in model_names}

for i, n in enumerate(N_PER_CLASS):
    for seed in range(N_SEEDS):
        idx = sample_labeled_subset(y_train, n, seed)
        for name in model_names:
            acc[name][i, seed] = fit_and_score(name, idx)

    summary_str = f"{n:>4} labeled/class ({len(class_names) * n:>5} total): "
    summary_str += "   ".join(f"{name}={acc[name][i].mean():.3f}" for name in model_names)
    print(summary_str)

In [ ]:
total_labels = [len(class_names) * n for n in N_PER_CLASS]
styles = {"w2v": "#eb6834", "tfidf": "#2a78d6", "random": "#8c8c8c"}

fig, ax = plt.subplots(figsize=(7.5, 5))
for name, label in model_names.items():
    mean, std = acc[name].mean(axis=1), acc[name].std(axis=1)
    ax.plot(total_labels, mean, color=styles[name], marker="o", linewidth=2, label=label)
    ax.fill_between(total_labels, mean - std, mean + std, color=styles[name], alpha=0.15)
ax.axhline(1 / len(class_names), color="black", linestyle=":", linewidth=1)
ax.text(total_labels[0], 1 / len(class_names) + 0.008, "chance", fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("total labeled training posts (log scale)")
ax.set_ylabel("test accuracy (20 classes)")
ax.set_title("Document classification vs. number of labeled examples\n(mean of 5 random labeled subsets, band = +/- 1 std)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
# reference point: what if we have every training label (11.3k posts)?
all_idx = np.arange(len(y_train))
print("accuracy with ALL training labels:")
for name, label in model_names.items():
    print(f"  {label:<40} {fit_and_score(name, all_idx):.3f}")

**Reading the plot and the numbers above**

- **Pretrained vs. random.** The Word2Vec curve sits above the random-embedding
  control at every label budget (0.498 vs 0.193 with 100 labeled posts per
  class). Both use the same averaging, the same 64 dimensions and the same
  classifier -- the only difference is what the encoder learned from
  unlabeled text. The averaging alone is worth very little.
- **Very few labels: pretrained vectors win.** With 1 to 5 labeled posts per
  class (20-100 posts in total) the pretrained vectors beat TF-IDF, e.g. 0.328
  vs 0.260 at 5 per class. A bag of words can only learn about a word from
  labeled posts that contain it, so with 20 labeled posts most of the ~5,000
  words never appear in a single training example (and the TF-IDF result
  swings widely depending on which posts happened to be drawn -- note its
  wide band). The word2vec vectors were shaped by unlabeled text: a post that
  mentions *nhl* lands near hockey posts even if no labeled post ever did.
- **Around 10 per class: a tie.** 0.384 vs 0.372 is within the run-to-run
  spread, so we cannot separate them.
- **More labels: the sparse model overtakes.** From 20 labeled posts per class
  onwards TF-IDF is ahead (0.566 vs 0.498 at 100 per class), and with all
  11.3k training labels it wins clearly (0.640 vs 0.546). It keeps the
  identity of every word, while the average squeezes a post into 64 numbers
  and discards word order. Once labels are plentiful, that capacity matters
  more than the prior knowledge the unlabeled text supplied.

So the pretrained word vectors are not a universal upgrade -- they are most
valuable in the regime this series keeps returning to: plentiful unlabeled
data, scarce labels.

### Conclusion

- Word2Vec is an autoencoder-shaped network -- a wide input, a narrow
  bottleneck, a wide output -- whose target is a word's **context** rather
  than the word itself. Trained on nothing but raw text, its bottleneck codes
  put words used in similar contexts close together (Section 7).
- Those codes are a **transferable asset when labels are very scarce**.
  Averaging the frozen word vectors of a post gives a 64-number document
  representation that, with a plain logistic regression on top, beats a
  TF-IDF bag of words when there are only a handful of labeled posts per
  class. The random-embedding control shows the gain comes from what was
  learned, not from the averaging (Section 8).
- The advantage is **limited to the low-label regime**. Around 10 labeled
  posts per class the two are roughly tied, and beyond that the sparse
  TF-IDF model overtakes the 64-d average -- by a wide margin once all the
  labels are used. Pretraining is not a free win: it pays off when labels
  are the bottleneck, and a strong sparse baseline should always be checked.
- The downstream recipe here is deliberately simple. Averaging discards word
  order and every word outside the small vocabulary, and the vectors come
  from only ~2M tokens of text; vectors pretrained on billions of words, or
  a model that reads the words in order, would move the crossover point far
  to the right.